In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder

from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from xgboost import XGBRegressor

import joblib

In [2]:
fusion = pd.read_csv("../dataset/processed/fusion_features.csv")

print(fusion.shape)

fusion.head()

(45493, 23)


,O2A_Load_Index,O2A_Rider_Index,FM_Travel_Index,FM_Vehicle_Index,WT_Delay_Index,LM_Delivery_Index,Order_Hour,Pickup_Hour,Weekend,Month,...,Traffic_Score,Weather_Score,Vehicle_Score,Vehicle_condition,multiple_deliveries,Rider_Experience,City,Festival,Type_of_order,Time_taken (min)
0,3.333333,635.04,41.122328,8,10,164.489313,21,22,1,2,...,4,4,4,2,3.0,151.2,Metropolitian,No,Snack,46
1,1.666667,463.89,18.726956,4,9,74.907824,14,15,1,2,...,3,5,4,1,1.0,98.7,Metropolitian,No,Meal,23
2,1.333333,508.07,27.575720,3,7,82.727161,17,17,0,3,...,2,6,3,1,1.0,108.1,Metropolitian,No,Drinks,21
3,0.333333,628.66,2.930258,0,7,11.721031,9,9,1,2,...,1,6,4,0,0.0,146.2,Metropolitian,No,Buffet,20
4,2.000000,530.16,77.586473,3,10,232.759419,19,20,0,2,...,4,4,3,1,1.0,112.8,Metropolitian,No,Snack,41


In [3]:
TARGET = "Time_taken (min)"

X = fusion.drop(columns=[TARGET])

y = fusion[TARGET]

print(X.shape)

(45493, 22)


In [4]:
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

numerical_features = X.select_dtypes(exclude=["object"]).columns.tolist()

print(categorical_features)

print()

print(numerical_features)

['City', 'Festival', 'Type_of_order']

['O2A_Load_Index', 'O2A_Rider_Index', 'FM_Travel_Index', 'FM_Vehicle_Index', 'WT_Delay_Index', 'LM_Delivery_Index', 'Order_Hour', 'Pickup_Hour', 'Weekend', 'Month', 'Delivery_person_Age', 'Delivery_person_Ratings', 'Trip_Distance_km', 'Traffic_Score', 'Weather_Score', 'Vehicle_Score', 'Vehicle_condition', 'multiple_deliveries', 'Rider_Experience']


In [5]:
numeric_transformer = Pipeline(

    steps=[

        ("imputer", SimpleImputer(strategy="median"))

    ]

)

categorical_transformer = Pipeline(

    steps=[

        ("imputer", SimpleImputer(strategy="most_frequent")),

        ("encoder", OneHotEncoder(handle_unknown="ignore"))

    ]

)

preprocessor = ColumnTransformer(

    transformers=[

        ("num", numeric_transformer, numerical_features),

        ("cat", categorical_transformer, categorical_features)

    ]

)

In [6]:
X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.20,

    random_state=42

)

print(X_train.shape)

print(X_test.shape)

(36394, 22)
(9099, 22)


In [7]:
xgb_pipeline = Pipeline(

    steps=[

        ("preprocessor", preprocessor),

        ("model",

         XGBRegressor(

             n_estimators=500,

             learning_rate=0.05,

             max_depth=8,

             subsample=0.8,

             colsample_bytree=0.8,

             objective="reg:squarederror",

             random_state=42

         ))

    ]

)

In [8]:
xgb_pipeline.fit(

    X_train,

    y_train

)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['O2A_Load_Index',
                                                   'O2A_Rider_Index',
                                                   'FM_Travel_Index',
                                                   'FM_Vehicle_Index',
                                                   'WT_Delay_Index',
                                                   'LM_Delivery_Index',
                                                   'Order_Hour', 'Pickup_Hour',
                                                   'Weekend', 'Month',
                                                   'Delivery_person_Age',
                                                   'Delivery_person_Ratings',
                                                   'Trip_Distanc...
                              feature_types=None, gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=0.05,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=8, max_leaves=None,
                              min_child_weight=None, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=500, n_jobs=None,
                              num_parallel_tree=None, random_state=42, ...))])

In [9]:
pred = xgb_pipeline.predict(

    X_test

)

In [10]:
mae = mean_absolute_error(

    y_test,

    pred

)

rmse = np.sqrt(

    mean_squared_error(

        y_test,

        pred

    )

)

r2 = r2_score(

    y_test,

    pred

)

print("="*40)

print("Fusion Architecture Performance")

print("="*40)

print(f"MAE  : {mae:.3f}")

print(f"RMSE : {rmse:.3f}")

print(f"R²   : {r2:.4f}")

Fusion Architecture Performance
MAE  : 3.135
RMSE : 3.952
R²   : 0.8232


In [11]:
joblib.dump(

    xgb_pipeline,

    "../modelv2/fusion_xgboost_eta.pkl"

)

print("Fusion Model Saved Successfully")

Fusion Model Saved Successfully


In [12]:
results = pd.DataFrame({

    "Model":[

        "Linear Regression",

        "Random Forest",

        "XGBoost",

        "Fusion XGBoost"

    ],

    "MAE":[

        4.77,

        3.10,

        3.05,

        mae

    ],

    "RMSE":[

        6.00,

        3.94,

        3.83,

        rmse

    ],

    "R2":[

        0.5927,

        0.8243,

        0.8336,

        r2

    ]

})

results

,Model,MAE,RMSE,R2
0,Linear Regression,4.77000,6.000000,0.592700
1,Random Forest,3.10000,3.940000,0.824300
2,XGBoost,3.05000,3.830000,0.833600
3,Fusion XGBoost,3.13496,3.951859,0.823242
